
# Module 6: 常時相談窓口と本番移行（20分）

## 目的
- dev/staging/prod のカタログ分離
- 簡単な Job / Lakeflow パイプライン
- Declarative Automation Bundles (DABs) での移行の型
- SDLC/AI チェックシート

## FE 制約
- ジョブ同時 5
- Lakeflow パイプライン各タイプ 1

In [0]:
%run ./config


## Step 1: dev / staging / prod カタログ分離

本番運用では環境ごとにカタログを分離します。

In [0]:
# 環境分離の実演
for env in ["dev", "staging", "prod"]:
    catalog_name = f"{CATALOG}_{env}"
    try:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{SCHEMA}")
        print(f"✅ {catalog_name}.{SCHEMA} 作成完了")
    except Exception as e:
        print(f"⚠️ {catalog_name}: {e}")


### 環境分離の意義

```
dev        →  開発者が自由に実験
staging    →  テスト・品質検証
prod       →  本番データ・参照専用
```

- 各環境で GRANT を分けることで、開発者が本番データを誤って変更するリスクを排除
- DABs で環境変数を切り替えることで同一コードを各環境にデプロイ


## Step 2: 簡単な Job 作成

ノートブックをジョブとしてスケジュール実行します。

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import Task, NotebookTask, CronSchedule
import os

w = WorkspaceClient()

# 現在のノートブックパスを取得
current_notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = "/".join(current_notebook.split("/")[:-1])

# ジョブ作成
try:
    job = w.jobs.create(
        name=f"workshop_daily_etl_{USER_SLUG}",
        tasks=[
            Task(
                task_key="env_check",
                notebook_task=NotebookTask(
                    notebook_path=f"{notebook_dir}/00_intro_env_check"
                )
            )
        ],
        schedule=CronSchedule(
            quartz_cron_expression="0 0 8 * * ?",  # 毎日 8:00
            timezone_id="Asia/Tokyo"
        )
    )
    print(f"✅ Job 作成完了: {job.job_id}")
    print(f"   名前: workshop_daily_etl_{USER_SLUG}")
    print(f"   スケジュール: 毎日 8:00 JST")
except Exception as e:
    print(f"⚠️ Job 作成: {e}")


## Step 3: Lakeflow パイプライン（概念）

Lakeflow Spark Declarative Pipelines (SDP) でデータパイプラインを定義:

```python
# 例: Bronze → Silver → Gold のパイプライン
import dlt

@dlt.table(comment="生ログ")
def bronze_vehicle_logs():
    return spark.readStream.table("vehicle_logs")

@dlt.table(comment="クレンジング済み")
@dlt.expect_or_drop("valid_battery", "battery_pct BETWEEN 0 AND 100")
def silver_vehicle_logs():
    return dlt.read_stream("bronze_vehicle_logs")

@dlt.table(comment="地域別集計")
def gold_location_summary():
    return dlt.read("silver_vehicle_logs").groupBy("location").agg(...)
```

> FE 制約: Lakeflow パイプラインは各タイプ 1 本まで


## Step 4: Declarative Automation Bundles (DABs) での移行

### DABs とは
- ノートブック・ジョブ・パイプラインを **YAML + ソースコード**で定義
- `databricks bundle deploy --target prod` で環境別デプロイ
- CI/CD パイプライン（GitHub Actions 等）と組み合わせ可能

### ディレクトリ構成例

```
my-project/
├── databricks.yml        # バンドル定義
├── resources/
│   ├── jobs.yml          # ジョブ定義
│   └── pipelines.yml    # パイプライン定義
├── src/
│   └── etl_notebook.py
└── tests/
```

```yaml
# databricks.yml
bundle:
  name: vehicle-analytics
targets:
  dev:
    default: true
    workspace:
      host: https://your-workspace.cloud.databricks.com
  prod:
    workspace:
      host: https://your-workspace.cloud.databricks.com
    variables:
      catalog: prod_catalog
```


## Step 5: SDLC / AI チェックシート

### 本番移行チェックリスト

| カテゴリ | 確認項目 | ツール |
| --- | --- | --- |
| データ品質 | スキーマ検証・データプロファイリング | Lakeflow Expectations |
| セキュリティ | RLS/マスク/ABAC 設定 | UC Governance |
| コスト | 予算アラート・タグ按分 | AI Gateway / FinOps |
| 可用性 | リトライ・アラート | Jobs 設定 |
| AI 固有 | ハルシネーションチェック・バイアス検証 | Guardrails / 評価 |
| 運用 | モニタリング・ログ | 推論テーブル |

### 製品/運用の線引き

| 層 | 内容 | 誰が担うか |
| --- | --- | --- |
| 製品機能 | UC, AI Gateway, Jobs, Apps | Databricks が提供 |
| 標準テンプレ | DABs, ノートブックテンプレ | Databricks + SI |
| 運用モデル | 代理申請・セキュア設計・基盤運用 | お客様 + SI |

→ Databricks が標準テンプレとガバナンス機能で下支えし、運用モデル層はお客様が設計


## クリーンアップ（任意）

In [0]:
# 作成したジョブを削除（必要な場合）
# w.jobs.delete(job_id=job.job_id)
# print(f"✅ Job {job.job_id} 削除")


## ✅ 完了条件

- [x] dev → prod のカタログ分離ができた
- [x] ジョブが 1 本動く
- [x] DABs での移行の型を理解した
- [x] SDLC/AI チェックシートの雛形を確認した